# BugBuster — Development Environment Setup

This notebook installs all prerequisites needed to build the **firmware** (ESP32-S3, PlatformIO) and the **desktop app** (Tauri + Leptos/WASM).

**Supports:** macOS, Windows, Linux

| Section | What it installs |
|---------|------------------|
| System check | Detects OS, existing tools |
| Rust toolchain | `rustup`, `rustc`, `cargo`, `wasm32-unknown-unknown` target |
| Trunk | WASM bundler for Leptos frontend |
| Tauri CLI | `cargo-tauri` for building the desktop app |
| PlatformIO | Firmware build system for ESP32-S3 |
| Node.js | Required by Tauri for frontend dev server |
| System libs | Platform-specific libraries (GTK/WebKit on Linux, Xcode tools on macOS) |
| Verification | Confirms everything is installed correctly |

**Run all cells in order.** Cells that detect an already-installed tool will skip installation.

In [ ]:
import subprocess
import sys
import os
import platform
import shutil
from pathlib import Path

# Detect system
OS = platform.system()  # 'Darwin', 'Windows', 'Linux'
ARCH = platform.machine()  # 'arm64', 'x86_64', 'AMD64'
IS_MAC = OS == 'Darwin'
IS_WIN = OS == 'Windows'
IS_LINUX = OS == 'Linux'

HOME = Path.home()
CARGO_BIN = HOME / '.cargo' / 'bin'

def which(cmd):
    """Find a command on PATH or in ~/.cargo/bin."""
    p = shutil.which(cmd)
    if p: return p
    cargo_path = CARGO_BIN / (cmd + ('.exe' if IS_WIN else ''))
    if cargo_path.exists(): return str(cargo_path)
    return None

def run(cmd, check=True, shell=False, **kwargs):
    """Run a command with live output."""
    print(f'$ {cmd if isinstance(cmd, str) else " ".join(cmd)}')
    r = subprocess.run(cmd, shell=shell, capture_output=False, text=True, **kwargs)
    if check and r.returncode != 0:
        print(f'  ⚠ Command failed (exit {r.returncode})')
    return r.returncode == 0

def run_quiet(cmd, shell=False):
    """Run a command and return (success, stdout)."""
    try:
        r = subprocess.run(cmd, shell=shell, capture_output=True, text=True, timeout=30)
        return r.returncode == 0, r.stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False, ''

def version_of(cmd, args=['--version']):
    """Get version string of a tool, checking PATH and ~/.cargo/bin."""
    path = which(cmd)
    if not path: return None
    ok, out = run_quiet([path] + args)
    return out if ok else None

print(f'System     : {OS} {ARCH}')
print(f'Python     : {sys.version.split()[0]}')
print(f'Home       : {HOME}')
print(f'Cargo bin  : {CARGO_BIN}')
print()

## 1. Pre-flight Check
Scans for all required tools and reports what's missing.

In [ ]:
tools = {
    'rustup':      version_of('rustup'),
    'rustc':       version_of('rustc'),
    'cargo':       version_of('cargo'),
    'trunk':       version_of('trunk'),
    'cargo-tauri': version_of('cargo', ['tauri', '--version']),
    'wasm-target': None,  # checked separately
    'platformio':  version_of('pio') or version_of('platformio'),
    'node':        version_of('node'),
    'npm':         version_of('npm'),
    'git':         version_of('git'),
}

# Check wasm32 target
rustup = which('rustup')
if rustup:
    ok, out = run_quiet([rustup, 'target', 'list', '--installed'])
    if ok and 'wasm32-unknown-unknown' in out:
        tools['wasm-target'] = 'installed'

print('═' * 50)
print('  Tool              Status')
print('═' * 50)
missing = []
for name, ver in tools.items():
    if ver:
        status = f'✅ {ver[:40]}'
    else:
        status = '❌ Not found'
        missing.append(name)
    print(f'  {name:<18} {status}')
print('═' * 50)

if not missing:
    print('\n🎉 All tools installed! Skip to the Verification cell.')
else:
    print(f'\n⚠ Missing: {', '.join(missing)}')
    print('  Run the cells below to install them.')

## 2. System Dependencies
Installs platform-specific build tools and libraries needed by Tauri.

In [ ]:
if IS_MAC:
    print('macOS: Checking Xcode Command Line Tools...')
    ok, _ = run_quiet(['xcode-select', '-p'])
    if ok:
        print('  ✅ Already installed')
    else:
        print('  Installing Xcode CLT (may open a dialog)...')
        run(['xcode-select', '--install'], check=False)

elif IS_LINUX:
    print('Linux: Installing Tauri system dependencies...')
    # Detect package manager
    if which('apt-get'):
        run(['sudo', 'apt-get', 'update'], check=False)
        run(['sudo', 'apt-get', 'install', '-y',
             'libwebkit2gtk-4.1-dev', 'build-essential', 'curl', 'wget',
             'file', 'libxdo-dev', 'libssl-dev', 'libayatana-appindicator3-dev',
             'librsvg2-dev', 'libgtk-3-dev', 'patchelf'], check=False)
    elif which('dnf'):
        run(['sudo', 'dnf', 'install', '-y',
             'webkit2gtk4.1-devel', 'openssl-devel', 'curl', 'wget',
             'file', 'libxdo-devel', 'libappindicator-gtk3-devel',
             'librsvg2-devel', 'gtk3-devel', 'patchelf'], check=False)
    elif which('pacman'):
        run(['sudo', 'pacman', '-Syu', '--noconfirm',
             'webkit2gtk-4.1', 'base-devel', 'curl', 'wget', 'file',
             'openssl', 'appmenu-gtk-module', 'gtk3', 'librsvg',
             'patchelf'], check=False)
    else:
        print('  ⚠ Unknown package manager. Install Tauri deps manually:')
        print('    https://v2.tauri.app/start/prerequisites/#linux')

elif IS_WIN:
    print('Windows: Checking Visual Studio Build Tools...')
    # Check for MSVC
    vs_path = Path(os.environ.get('ProgramFiles(x86)', '')) / 'Microsoft Visual Studio'
    if vs_path.exists():
        print('  ✅ Visual Studio found')
    else:
        print('  ⚠ Visual Studio Build Tools not found.')
        print('  Download from: https://visualstudio.microsoft.com/visual-cpp-build-tools/')
        print('  Select "Desktop development with C++" workload.')

print('\n✅ System dependencies check complete.')

## 3. Rust Toolchain
Installs `rustup`, `rustc`, `cargo`, and the `wasm32-unknown-unknown` target.

In [ ]:
if which('rustup'):
    print(f'✅ rustup already installed: {version_of("rustup")}')
else:
    print('Installing Rust via rustup...')
    if IS_WIN:
        print('  Downloading rustup-init.exe...')
        run(['powershell', '-Command',
             'Invoke-WebRequest -Uri https://win.rustup.rs/x86_64 -OutFile rustup-init.exe; .\\rustup-init.exe -y'],
            shell=True, check=False)
    else:
        run('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y',
            shell=True, check=False)
    # Add to PATH for this session
    os.environ['PATH'] = str(CARGO_BIN) + os.pathsep + os.environ.get('PATH', '')
    print(f'  rustc: {version_of("rustc")}')

# Ensure stable toolchain
rustup = which('rustup')
if rustup:
    run([rustup, 'default', 'stable'], check=False)
    run([rustup, 'update'], check=False)

# Add wasm32 target
if rustup:
    ok, out = run_quiet([rustup, 'target', 'list', '--installed'])
    if ok and 'wasm32-unknown-unknown' in out:
        print('✅ wasm32-unknown-unknown target already installed')
    else:
        print('Adding wasm32-unknown-unknown target...')
        run([rustup, 'target', 'add', 'wasm32-unknown-unknown'])

print(f'\nrustc  : {version_of("rustc")}')
print(f'cargo  : {version_of("cargo")}')

## 4. Trunk (WASM Bundler)
Installs `trunk` which builds the Leptos frontend to WebAssembly.

In [ ]:
if which('trunk'):
    print(f'✅ trunk already installed: {version_of("trunk")}')
else:
    print('Installing trunk...')
    cargo = which('cargo')
    if cargo:
        run([cargo, 'install', 'trunk', '--locked'])
    else:
        print('  ⚠ cargo not found — install Rust first')

## 5. Tauri CLI
Installs `cargo-tauri` which builds the native desktop app shell.

In [ ]:
tauri_ver = version_of('cargo', ['tauri', '--version'])
if tauri_ver:
    print(f'✅ cargo-tauri already installed: {tauri_ver}')
else:
    print('Installing cargo-tauri...')
    cargo = which('cargo')
    if cargo:
        run([cargo, 'install', 'tauri-cli', '--locked'])
    else:
        print('  ⚠ cargo not found — install Rust first')

## 6. Node.js
Required by Tauri's dev server. Installs via the platform's preferred method if missing.

In [ ]:
if which('node'):
    print(f'✅ node already installed: {version_of("node")}')
else:
    print('Installing Node.js...')
    if IS_MAC:
        if which('brew'):
            run(['brew', 'install', 'node'], check=False)
        else:
            print('  ⚠ Homebrew not found. Install Node.js from https://nodejs.org/')
    elif IS_LINUX:
        if which('apt-get'):
            run('curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -',
                shell=True, check=False)
            run(['sudo', 'apt-get', 'install', '-y', 'nodejs'], check=False)
        elif which('dnf'):
            run(['sudo', 'dnf', 'install', '-y', 'nodejs'], check=False)
        elif which('pacman'):
            run(['sudo', 'pacman', '-S', '--noconfirm', 'nodejs', 'npm'], check=False)
        else:
            print('  ⚠ Install Node.js from https://nodejs.org/')
    elif IS_WIN:
        if which('winget'):
            run(['winget', 'install', 'OpenJS.NodeJS.LTS'], check=False)
        elif which('choco'):
            run(['choco', 'install', 'nodejs-lts', '-y'], check=False)
        else:
            print('  ⚠ Install Node.js from https://nodejs.org/')

## 7. PlatformIO
Installs PlatformIO Core (CLI) for building and flashing ESP32 firmware.

In [ ]:
pio_path = which('pio') or which('platformio')

# Also check the default PlatformIO venv location
if not pio_path:
    pio_venv = HOME / '.platformio' / 'penv' / 'bin' / 'pio'
    if IS_WIN:
        pio_venv = HOME / '.platformio' / 'penv' / 'Scripts' / 'pio.exe'
    if pio_venv.exists():
        pio_path = str(pio_venv)

if pio_path:
    ok, ver = run_quiet([pio_path, '--version'])
    print(f'✅ PlatformIO already installed: {ver}')
else:
    print('Installing PlatformIO Core...')
    # Use the official installer script
    if IS_WIN:
        run('pip install platformio', shell=True, check=False)
    else:
        run('curl -fsSL -o get-platformio.py https://raw.githubusercontent.com/platformio/platformio-core-installer/master/get-platformio.py && python3 get-platformio.py',
            shell=True, check=False)
    # Verify
    pio_venv = HOME / '.platformio' / 'penv' / 'bin' / 'pio'
    if pio_venv.exists():
        print(f'  ✅ Installed at {pio_venv}')
    else:
        print('  ⚠ PlatformIO installation may have failed.')
        print('  Try: pip install platformio')

# Install ESP32-S3 platform (pre-download so first build is faster)
if pio_path or (HOME / '.platformio' / 'penv' / 'bin' / 'pio').exists():
    pio = pio_path or str(HOME / '.platformio' / 'penv' / 'bin' / 'pio')
    print('\nPre-installing ESP32 platform (this may take a few minutes)...')
    run([pio, 'pkg', 'install', '-g', '-p', 'espressif32@6.9.0'], check=False)

## 8. Python Dependencies
Installs Python packages used by the notebooks (pyserial for flashing/monitoring).

In [ ]:
py_deps = ['pyserial', 'Pillow']

for pkg in py_deps:
    try:
        __import__(pkg.lower().replace('-', '_').split('[')[0])
        print(f'  ✅ {pkg} already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                      capture_output=True)
        print(f'  ✅ {pkg} installed')

## 9. Final Verification
Re-checks all tools and confirms the environment is ready.

In [ ]:
# Refresh PATH to pick up newly installed tools
os.environ['PATH'] = str(CARGO_BIN) + os.pathsep + os.environ.get('PATH', '')

checks = [
    ('Rust compiler',    'rustc',      ['--version']),
    ('Cargo',            'cargo',      ['--version']),
    ('Trunk',            'trunk',      ['--version']),
    ('Tauri CLI',        'cargo',      ['tauri', '--version']),
    ('Node.js',          'node',       ['--version']),
    ('npm',              'npm',        ['--version']),
    ('Git',              'git',        ['--version']),
]

# PlatformIO (special path)
pio_cmd = which('pio') or str(HOME / '.platformio' / 'penv' / 'bin' / 'pio')

print('╔' + '═' * 58 + '╗')
print('║  BugBuster Development Environment — Final Check         ║')
print('╠' + '═' * 58 + '╣')

all_ok = True
for label, cmd, args in checks:
    path = which(cmd)
    if path:
        ok, ver = run_quiet([path] + args)
        ver_short = ver.split('\n')[0][:35] if ver else ''
        print(f'║  ✅ {label:<20} {ver_short:<33}║')
    else:
        print(f'║  ❌ {label:<20} {"NOT FOUND":<33}║')
        all_ok = False

# PlatformIO
if Path(pio_cmd).exists():
    ok, ver = run_quiet([pio_cmd, '--version'])
    ver_short = ver.split('\n')[0][:35] if ver else ''
    print(f'║  ✅ {"PlatformIO":<20} {ver_short:<33}║')
else:
    print(f'║  ❌ {"PlatformIO":<20} {"NOT FOUND":<33}║')
    all_ok = False

# WASM target
rustup = which('rustup')
if rustup:
    ok, out = run_quiet([rustup, 'target', 'list', '--installed'])
    if ok and 'wasm32-unknown-unknown' in out:
        print(f'║  ✅ {"WASM target":<20} {"wasm32-unknown-unknown":<33}║')
    else:
        print(f'║  ❌ {"WASM target":<20} {"NOT INSTALLED":<33}║')
        all_ok = False

print('╠' + '═' * 58 + '╣')
if all_ok:
    print('║  🎉 Environment ready! You can now build BugBuster.     ║')
    print('║                                                          ║')
    print('║  Firmware:  Open esp32_flash.ipynb → Flash Firmware      ║')
    print('║  App:       Open run_app.ipynb → Dev mode                ║')
else:
    print('║  ⚠ Some tools are still missing. Re-run failed cells.   ║')
print('╚' + '═' * 58 + '╝')